In [ ]:
# import common libs
from pyspark.sql import SparkSession
import pandas as pd
import numpy as np
from pyspark.sql.types import *

from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

# Import the evaluation submodule
import pyspark.ml.evaluation as evals

# Import LogisticRegression
from pyspark.ml.classification import LogisticRegression

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession \
        .builder \
        .master("local[*]") \
        .config("spark.executor.memory", "16g") \
        .config("spark.driver.memory", "16g") \
        .config("spark.memory.offHeap.enabled",True) \
        .config("spark.memory.offHeap.size","16g") \
        .appName("Airline") \
        .getOrCreate()


        # .config("spark.sql.shuffle.partitions",60) \

In [ ]:
flights_df = spark.read.option("header",True).csv('Data_Dump/data/flights/2013.csv')
flights_df.createOrReplaceTempView("Flights")
flights_df=flights_df.select([when(col(c).contains('None') | \
                            col(c).contains('NULL') | \
                            (col(c) == '' ) | \
                            col(c).isNull() | \
                            isnan(c),0).otherwise(col(c)).alias(c) for c in flights_df.columns])
#checking if the file has any NaN values
flights_df.select([count(when(isnan(c) | col(c).isNull(), c)).alias(c) for c in flights_df.columns]).show()


In [ ]:
from pyspark.sql.functions import *
 
df1 = flights_df.select(col('ORIGIN'), \
            col('DEST'), \
            col('OP_CARRIER'), \
            col('ARR_DELAY'),to_date(flights_df.FL_DATE, 'yyyy-MM-dd').alias('FL_DATE'))

In [ ]:
fin=df1.withColumn('MONTH', month(df1.FL_DATE)).withColumn('DAY', month(df1.FL_DATE)).drop('FL_DATE')

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder



le_dest = LabelEncoder()

fl_df=fin.toPandas()
le_carrier = LabelEncoder()
fl_df['OP_CARRIER']= le_carrier.fit_transform(fl_df['OP_CARRIER'])
le_origin = LabelEncoder()
fl_df['ORIGIN'] = le_origin.fit_transform(fl_df['ORIGIN'])
le_dest = LabelEncoder()
fl_df['DEST'] = le_dest.fit_transform(fl_df['DEST'])


In [ ]:
# fl_df=fin.toPandas()
# df = pd.concat([fl_df,pd.get_dummies(fl_df['OP_CARRIER'],drop_first=True,prefix="OP_CARRIER")],axis=1)
# df = pd.concat([fl_df,pd.get_dummies(fl_df['ORIGIN'],drop_first=True,prefix="ORIGIN")],axis=1)
# df = pd.concat([fl_df,pd.get_dummies(fl_df['DEST'],drop_first=True,prefix="DEST")],axis=1)
# df = pd.concat([fl_df,pd.get_dummies(fl_df['DAY'],drop_first=True,prefix="DAY")],axis=1)
# df = pd.concat([fl_df,pd.get_dummies(fl_df['MONTH'],drop_first=True,prefix="MONTH")],axis=1)
# df.drop(['ORIGIN','DEST','OP_CARRIER','DAY','MONTH'],axis=1,inplace=True)


In [ ]:
fl_df

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(fl_df.drop('ARR_DELAY',axis=1),fl_df['ARR_DELAY'], test_size=0.30)


In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
from sklearn.preprocessing import LabelEncoder

Dtc = DecisionTreeRegressor(random_state = 2)
Dtc.fit(X_train,y_train)
predictedValues = Dtc.predict(X_test)
print ('MAE:' ,  mean_absolute_error(y_test, predictedValues))
print ('MSE:' , mean_squared_error(y_test, predictedValues))
print('RMSE:' , np.sqrt(mean_squared_error(y_test, predictedValues)))
print ('R2:' , r2_score(y_test, predictedValues))



In [ ]:
X_test.head() 1307321

In [ ]:
from sklearn.preprocessing import LabelEncoder

x1 = [m,d]
x2 = [c, o, dest]
x1.extend(x2)
df1 = pd.DataFrame(data = [x1], columns = ['MONTH','DAY', 'OP_CARRIER', 'ORIGIN', 'DEST'])
#df1['OP_CARRIER'] = le_carrier.fit(df1['OP_CARRIER'])
df1['OP_CARRIER'] = le_carrier.transform(df1['OP_CARRIER'])
df1['ORIGIN'] = le_origin.transform(df1['ORIGIN'])
df1['DEST'] = le_dest.transform(df1['DEST'])
x = df1.iloc[:, :5].values
ans = Dtc.predict(x)
output = ans

In [ ]:
print("the flight is delayed for: ", str(output[0]))

In [ ]:

from tkinter import *
 
top = Tk()  
top.geometry("250x250") 
def printValue():
    m = month.get()
    d=day.get()
    c=carrier.get()
    o=origin.get()
    dest=desti.get()
    x1 = [m,d]
    x2 = [c, o, dest]
    x1.extend(x2)
    df1 = pd.DataFrame(data = [x1], columns = ['MONTH','DAY', 'OP_CARRIER', 'ORIGIN', 'DEST'])
    #df1['OP_CARRIER'] = le_carrier.fit(df1['OP_CARRIER'])
    df1['OP_CARRIER'] = le_carrier.transform(df1['OP_CARRIER'])
    df1['ORIGIN'] = le_origin.transform(df1['ORIGIN'])
    df1['DEST'] = le_dest.transform(df1['DEST'])
    x = df1.iloc[:, :5].values
    ans = Dtc.predict(x)
    output = ans
    Label(top, text=f'{output}, is your delay!', bg='#ffbf00').grid(row=13, column=1, sticky=W) 
# the label for user_name

   # the label for user_name
# month_lbl = Label(top,text = "month in number").place(x = 40,y = 60) 
   
# # the label for user_password 
# day_lbl = Label(top,text = "day in number").place(x = 40, y = 100)
# carrier_lbl = Label(top,text = "Carrier").place(x = 40,y = 140) 
   
# # the label for user_password 
# origin_lbl = Label(top,text = "Origin").place(x = 40, y = 180)
# dest_lbl = Label(top,text = "Destination").place(x = 40, y = 220)
month = Entry(top,width = 30)
month.grid(row=1, column=1, sticky=W)
month.insert(0, 'month in numbers')
day = Entry(top,width = 30)
day.grid(row=3, column=1, sticky=W)
day.insert(0, 'day of month in numbers')
carrier = Entry(top,width = 30)
carrier.grid(row=5, column=1, sticky=W)
carrier.insert(0, 'carrier')
origin = Entry(top,width = 30)
origin.grid(row=7, column=1, sticky=W)
origin.insert(0, 'origin')
desti = Entry(top,width = 30)
desti.grid(row=9, column=1, sticky=W)
desti.insert(0, 'dest')

# day=Entry(top,width = 30).place(x = 110,y = 100)
# carrier=Entry(top,width = 30).place(x = 110,y = 140)
# origin=Entry(top,width = 30).place(x = 110,y = 180)
# destination=Entry(top,width = 30).place(x = 110,y = 220)
submit_button = Button(top,text = "Submit", bg="green", command=printValue).grid(row=11, column=1, sticky=W)
top.mainloop()